In [4]:
import json
import glob
import os
from pathlib import Path
import time
import os
from dotenv import load_dotenv

load_dotenv()

def consolidar_jsons(fonte, cidade, PASTA_DADOS):
    
    now = time.strftime("%Y-%m")
    
    padrao_busca = str(PASTA_DADOS / f'{cidade}_{fonte}_*.json')
    
    arquivos_json = glob.glob(padrao_busca)
    
    if not arquivos_json:
        print("Nenhum arquivo encontrado com o padrão especificado.")
        return

    dados_consolidados = []
    total_arquivos = len(arquivos_json)

    print(f"Iniciando a união de {total_arquivos} arquivos...")

    for i, caminho in enumerate(arquivos_json, 1):
        nome_base = os.path.basename(caminho)
        with open(caminho, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
                # Verifica se o conteúdo é uma lista (padrão do seu scraper)
                if isinstance(conteudo, list):
                    dados_consolidados.extend(conteudo)
                else:
                    dados_consolidados.append(conteudo)
                
                print(f"[{i}/{total_arquivos}] Adicionado: {nome_base} ({len(conteudo)} itens)")
            except Exception as e:
                print(f"Erro ao ler {nome_base}: {e}")

    # 2. Salva o arquivo final consolidado
    nome_final = f'{cidade}_{fonte}_{now}.json'
    
    caminho_final = PASTA_DADOS / nome_final

    try:
        with open(caminho_final, 'w', encoding='utf-8') as f_out:
            json.dump(dados_consolidados, f_out, indent=4, ensure_ascii=False)
        
        # --- VALIDAÇÃO DE SEGURANÇA ---
        tamanho_final = os.path.getsize(caminho_final)
        
        """if tamanho_final > 0 and len(dados_consolidados) > 0:
            print(f"✅ Consolidação concluída: {len(dados_consolidados)} registros.")
            print(f"📦 Arquivo gerado: {nome_final} ({tamanho_final / 1024 / 1024:.2f} MB)")
            
            # 4. Deleta os arquivos anteriores apenas se o final estiver OK
           
            print("🗑️ Removendo arquivos temporários (fatias)...")
            for arquivo_velho in arquivos_json:
                try:
                    os.remove(arquivo_velho)
                    print(f"   Excluído: {os.path.basename(arquivo_velho)}")
                except Exception as e:
                    print(f"   Erro ao excluir {arquivo_velho}: {e}")
        
        
            
            print("✨ Limpeza concluída com sucesso!")
        else:
            print("⚠️ Erro crítico: O arquivo final parece estar vazio. Abortando exclusão.")"""

    except Exception as e:
        print(f"❌ Erro ao salvar arquivo consolidado: {e}")


In [5]:

import asyncio
import sys
from pathlib import Path
import time

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / 'balneario_camboriu'

#consolidar_jsons('olx', 'balneario_camboriu', PASTA_DADOS)


In [6]:
PASTA_DADOS

WindowsPath('c:/Users/jefer/Documents/Ciencia-de-dados/Preco-Imoveis/dados/balneario_camboriu')

In [7]:
import pandas as pd
import warnings
import logging
import asyncio
from funcoes_limpando_dados_imoveis import (limpar_valor_iptu,
                                            limpar_banheiros, 
                                            limpar_metragem, 
                                            limpar_vagas,  
                                            #limpa_endereco_apply, 
                                            limpar_valor_condominio, 
                                            converter_para_data, 
                                            classificar_tipo_imovel, 
                                            reclassificar_outros, 
                                            preencher_todas_coordenadas,
                                            main_example, 
                                            limpar_valor_venda, 
                                            limpar_quartos, 
                                            pirabeiraba_dona_francisca, 
                                            geocodificar_dataframe,
                                            limpa_endereco_apply_zap, 
                                            limpa_endereco_apply_chave_mao, 
                                            limpa_endereco_apply_olx,)
import time
from datetime import datetime
import json
from pathlib import Path

import asyncio
import sys
from pathlib import Path
import time

cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / cidade

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)

warnings.filterwarnings("ignore")

start_time = time.time()

async def limpando_dados_cidades(pd_data, batch, cidade_limpeza = 'joinville', estado_limpeza = 'sc', cidade_localizacao = 'Joinville', estado_localizacao = 'SC',  tipo_async=True,  pais='Brasil'): 
       
    logger.info("Iniciando o processo de limpeza de dados de imóveis...")    
    
    pd_data = pd_data.drop_duplicates(subset=['url'])

    pd_data = pd_data[pd_data['valor_imovel'].notna()]

    pd_data_sem_nulos = pd_data.dropna(thresh=10)

    logger.info(f"Removendo linhas com muitos valores faltantes. Registros restantes: {pd_data_sem_nulos.shape}")

    #endereco_dividido = pd_data_sem_nulos['endereco'].apply(limpa_endereco_apply)

    #pd_data_endereco_dividido = pd.concat([pd_data_sem_nulos, endereco_dividido], axis=1).drop('endereco', axis=1)

    #pd_data_endereco_dividido['bairro'] = pd_data_endereco_dividido['bairro'].apply(pirabeiraba_dona_francisca)

    #logger.info("Coluna 'endereco' dividida em 'rua', 'bairro', 'cidade' e 'estado'...")
    
    pd_data_estado = pd_data_sem_nulos.copy()

    pd_data_estado = pd_data_estado[pd_data_estado['estado'] == estado_limpeza]

    logger.info(f"Removendo linhas com estado diferente de {estado_limpeza}. Registros restantes: {pd_data_estado.shape}")

    pd_data_metragem = pd_data_estado.copy()

    pd_data_metragem['metragem'] = pd_data_metragem['metragem'].apply(limpar_metragem)

    logger.info(f"Coluna 'metragem' limpa. Registros restantes: {pd_data_metragem.shape}")

    pd_data_valor_imovel = pd_data_metragem.copy()

    try:
        pd_data_valor_imovel['valor_venda'] = pd_data_valor_imovel['valor_venda'].apply(limpar_valor_venda)
    except:
        pd_data_valor_imovel['valor_imovel'] = pd_data_valor_imovel['valor_imovel'].apply(limpar_valor_venda)


    logger.info(f"Coluna 'valor_venda' limpa. Registros restantes: {pd_data_valor_imovel.shape}")

    pd_data_valor_condominio = pd_data_valor_imovel.copy()

    pd_data_valor_condominio['condominio'] = pd_data_valor_condominio['condominio'].apply(limpar_valor_condominio)

    logger.info(f"Coluna 'condominio' limpa. Registros restantes: {pd_data_valor_condominio.shape}")

    pd_data_valor_iptu = pd_data_valor_condominio.copy()

    pd_data_valor_iptu['iptu'] = pd_data_valor_iptu['iptu'].apply(limpar_valor_iptu)
    

    logger.info(f"Coluna 'iptu' limpa. Registros restantes: {pd_data_valor_iptu.shape}")

    pd_data_ano_publicacao = pd_data_valor_iptu.copy()

    pd_data_ano_publicacao['data_criacao'] = pd_data_ano_publicacao['data_criacao'].apply(converter_para_data)

    pd_data_ano_publicacao['dias_publicacao'] = (pd.to_datetime(datetime.now().strftime('%Y-%m-%d')) - pd.to_datetime(pd_data_ano_publicacao['data_criacao'], format='%d/%m/%Y')).dt.days
    
    logger.info(f"Coluna 'data_criacao' limpa. Registros restantes: {pd_data_ano_publicacao.shape}")

    pd_data_banheiros = pd_data_ano_publicacao.copy()
    
    pd_data_banheiros['banheiros'] = pd_data_banheiros['banheiros'].apply(limpar_banheiros)

    logger.info(f"Coluna 'banheiros' limpa. Registros restantes: {pd_data_banheiros.shape}")

    pd_data_quartos = pd_data_banheiros.copy()

    pd_data_quartos['quartos'] = pd_data_quartos['quartos'].apply(limpar_quartos)

    logger.info(f"Coluna 'quartos' limpa. Registros restantes: {pd_data_quartos.shape}")

    pd_data_garagem = pd_data_quartos.copy()

    #pd_data_garagem['vagas'] = pd_data_garagem['vagas'].replace('--', 0).astype('int64')

    pd_data_garagem['vagas'] = pd_data_garagem['vagas'].apply(limpar_vagas)

    logger.info(f"Coluna 'vagas' limpa. Registros restantes: {pd_data_garagem.shape}")

    pd_data_tipo_imovel = pd_data_garagem.copy()

    pd_data_tipo_imovel['tipo_imovel'] = pd_data_tipo_imovel['titulo'].apply(classificar_tipo_imovel)

    mask = pd_data_tipo_imovel['tipo_imovel'] == 'outros'

    pd_data_tipo_imovel.loc[mask, 'tipo_imovel'] = (
        pd_data_tipo_imovel.loc[mask, 'descricao']
        .apply(reclassificar_outros)
    )

    logger.info(f"Coluna 'tipo_imovel' classificada. Registros restantes: {pd_data_tipo_imovel.shape}")

    pd_data_long_lat = pd_data_tipo_imovel.copy()
    
    if tipo_async:
        pd_data_lat_log_completo = await preencher_todas_coordenadas(pd_data_long_lat, batch_size=batch, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)
    else:
        pd_data_lat_log_completo = geocodificar_dataframe(pd_data_long_lat, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)

    logger.info(f"Todas as coordenadas preenchidas. Registros restantes: {pd_data_lat_log_completo.shape}")

    pd_data_lat_log_completo['preco_por_m2'] = pd_data_lat_log_completo['valor_imovel'] / pd_data_lat_log_completo['metragem']

    logger.info(f'Coluna "preco_por_m2" criada. Registros restantes: {pd_data_lat_log_completo.shape}')

    def classificar_dentro_bairro(grupo):
        p25 = grupo["preco_por_m2"].quantile(0.25)
        p50 = grupo["preco_por_m2"].quantile(0.50)
        p75 = grupo["preco_por_m2"].quantile(0.75)
        
        def faixa(val):
            if val <= p25:
                return "barato"
            elif val <= p50:
                return "medio_baixo"
            if val <= p75:
                return "medio_alto"
            else:
                return "alto_padrao"

        grupo = grupo.copy()
        grupo["faixa"]       = grupo["preco_por_m2"].apply(faixa)
        grupo["p25_bairro"]  = p25
        grupo["p50_bairro"]  = p50
        grupo["p75_bairro"]  = p75
        return grupo

    pd_data_range_bairro_tipo_imovel = pd_data_lat_log_completo.groupby(["bairro", "tipo_imovel"], group_keys=False, ).apply(classificar_dentro_bairro)

    pd_data_range_bairro_tipo_imovel = pd.concat([pd_data_lat_log_completo, pd_data_range_bairro_tipo_imovel[['faixa', 'p25_bairro', 'p50_bairro', 'p75_bairro']]], axis=1)

    logger.info(f"Criando Faixas de preço por bairro e tipo de imóvel classificadas. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")

    pd_data_range_bairro_tipo_imovel["desvio_mediana"] = round((pd_data_range_bairro_tipo_imovel["preco_por_m2"] - pd_data_range_bairro_tipo_imovel["p50_bairro"]) / pd_data_range_bairro_tipo_imovel["p50_bairro"],2)

    logger.info(f"Coluna 'desvio_mediana' criada. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")
    
    return pd_data_range_bairro_tipo_imovel

In [8]:
def carregar_json(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo mais recente pelo padrão e retorna um DataFrame.
    Retorna DataFrame vazio se não encontrar nenhum arquivo.
    """
    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    
    logger.info(f"Arquivo encontrado: {arquivo.name}")

    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return pd.DataFrame(data), arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")
        
def carregar_parquet(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo .parquet mais recente pelo padrão e retorna um DataFrame.
    """
    # Garante que estamos buscando arquivos .parquet se o pattern não especificar
    if not glob_pattern.endswith('.parquet'):
        glob_pattern = glob_pattern.replace('.json', '.parquet')

    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    # Ordena para pegar o mais recente (mantendo sua lógica de data no final do nome)
    try:
        arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    except Exception:
        arquivo = max(arquivos, key=lambda f: f.stat().st_mtime) # Fallback para data de modificação
    
    logger.info(f"Arquivo Parquet encontrado: {arquivo.name}")

    try:
        # No Parquet, o pandas lê o arquivo diretamente pelo caminho
        df = pd.read_parquet(arquivo)
        return df, arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    # Esta função permanece igual, pois Path.unlink() deleta qualquer tipo de arquivo
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")

In [9]:
def normalizar_bairros(bairro, mapeamento):
    if not isinstance(bairro, str):
        return bairro
        
    bairro_low = bairro.lower()
    
    for nome_correto, variacoes in mapeamento.items():
        # Verifica se qualquer uma das variações está contida no nome original
        if any(v in bairro_low for v in variacoes):
            return nome_correto
            
    return bairro 

async def limpando_dados(name_arquivo_zap : str, 
         name_arquivo_vivareal: str, 
         name_arquivo_chave_mao: str,
         name_arquivo_olx: str,
         name_arquivo_saida: str,
         pasta_dados : Path, 
         batch: int = 1,
         tipo_async: bool = False,
         cidade_localizacao: str = 'Joinville', 
         cidade_limpeza: str = 'joinville',
         estado_limpeza: str = 'sc', 
         estado_localizacao: str = 'SC',
         pais: str = 'Brasil', 
         MAPA_BAIRROS: dict = None,):
    
    logger.info(f"Iniciando limpeza de dados de imóveis de {cidade_limpeza}...")
    
    logger.info(f"Pasta de dados: {pasta_dados}")

    pasta_dados.mkdir(parents=True, exist_ok=True)

    df_zap, arquivo_zap      = carregar_json(pasta_dados, name_arquivo_zap)
    
    df_vivareal, arquivo_vivareal = carregar_json(pasta_dados,name_arquivo_vivareal)
    
    df_chave_mao, arquivo_chave_mao = carregar_json(pasta_dados,name_arquivo_chave_mao)
    
    df_olx, arquivo_olx = carregar_json(pasta_dados,name_arquivo_olx)

    if not df_zap.empty:
        df_zap['fonte'] = 'zap_imoveis'
    if not df_vivareal.empty:
        df_vivareal['fonte'] = 'viva_real'

    if not df_chave_mao.empty:
        df_chave_mao['fonte'] = 'chave_mao'
    
    if not df_olx.empty:
        df_olx['fonte'] = 'olx'

    if df_zap.empty and df_vivareal.empty and df_chave_mao.empty and df_olx.empty:
        logger.error("Nenhum dado encontrado em nenhuma das fontes — abortando.")
        return
    
    #df_zap_endereco_limpo = df_zap['endereco'].apply(limpa_endereco_apply_zap)
    #df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(limpa_endereco_apply_zap)
    #df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(limpa_endereco_apply_chave_mao)
    #df_olx_endereco_limpo = df_olx['endereco'].apply(limpa_endereco_apply_olx)
    
    df_zap_endereco_limpo = df_zap['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(lambda x: limpa_endereco_apply_chave_mao(x, cidade_limpeza, estado_limpeza))
    df_olx_endereco_limpo = df_olx['endereco'].apply(lambda x: limpa_endereco_apply_olx(x, cidade_limpeza, estado_limpeza))
    
    df_zap_endereco =  pd.concat([df_zap, df_zap_endereco_limpo], axis=1)
    df_vivareal_endereco =  pd.concat([df_vivareal, df_vivareal_endereco_limpo], axis=1)
    df_chave_mao_endereco =  pd.concat([df_chave_mao, df_chave_mao_endereco_limpo], axis=1)
    df_olx_endereco =  pd.concat([df_olx, df_olx_endereco_limpo], axis=1)

    df = pd.concat([df_zap_endereco if not df_zap_endereco.empty else pd.DataFrame(), 
                    df_vivareal_endereco if not df_vivareal_endereco.empty else pd.DataFrame(),
                    df_chave_mao_endereco if not df_chave_mao_endereco.empty else pd.DataFrame(),
                    df_olx_endereco if not df_olx_endereco.empty else pd.DataFrame()], 
                   axis=0, ignore_index=True)
    
    logger.info(f"Total de registros carregados: {len(df)} (zap: {len(df_zap)} | vivareal: {len(df_vivareal)} | chave_mao: {len(df_chave_mao)} | olx: {len(df_olx)})")

    # Limpeza
    df_limpo = await limpando_dados_cidades(df, 
                                        batch = batch, 
                                        cidade_limpeza= cidade_limpeza, 
                                        cidade_localizacao= cidade_localizacao,
                                        tipo_async=tipo_async,
                                        estado_limpeza= estado_limpeza,
                                        estado_localizacao= estado_localizacao, 
                                        pais= pais
                                        )

    # Remove duplicatas
    colunas_dedup = ['valor_imovel', 'rua', 'bairro', 'metragem', 'quartos', 'preco_por_m2', 'banheiros', 'lat', 'lng']
    
    colunas_dedup = [c for c in colunas_dedup if c in df_limpo.columns]  

    antes = len(df_limpo)
    
    df_limpo = df_limpo.drop_duplicates(subset=colunas_dedup, keep='first').reset_index(drop=True)
    
    logger.info(f"Duplicatas removidas: {antes - len(df_limpo)} | Registros finais: {len(df_limpo)}")
    
    if MAPA_BAIRROS:
        df_limpo['bairro'] = df_limpo['bairro'].apply(normalizar_bairros, args=(MAPA_BAIRROS,))
    
    logger.info(f"Coluna 'bairro' corrigida...")
    
    #df_limpo = df_limpo.groupby('bairro').filter(lambda x: len(x) > 1)
    
    return df_limpo




In [10]:
cidade = 'balneario camboriu'
cidade_limpeza = 'balneario camboriu'
estado_limpeza = 'sc'

cidade_localizacao = 'Balneario Camboriu'
estado_localizacao = 'SC'

pais = 'Brasil'

BATCH = 100

PASTA_DADOS  = Path().absolute().parent/ 'dados'/ cidade

MAPA_BAIRROS = {
        'pirabeiraba': ['pirabeiraba', 'dona francisca', 'distrito industrial norte'],
        'distrito industrial': ['distrito industrial', 'zona industrial', 'distrito industrial norte', 'distrito industrial sul'],
        'area rural': ['area rural', 'rural'],
        'vila nova': ['vila nova'],   
        's/b': ['localizacao'],
    }

"""result = await limpando_dados(name_arquivo_zap = f'{cidade}_zap_*.json', 
               name_arquivo_vivareal = f'{cidade}_vivareal_*.json', 
               name_arquivo_chave_mao = f'{cidade}_chave_mao_*.json',
               name_arquivo_olx = f'{cidade}_olx_*.json',
               name_arquivo_saida = f'{cidade}_imoveis_limpo', 
               pasta_dados = PASTA_DADOS, 
               tipo_async = True,
               batch = BATCH, 
               cidade_limpeza=cidade_limpeza,
               cidade_localizacao=cidade_localizacao,
               estado_limpeza=estado_limpeza,
               estado_localizacao=estado_localizacao, 
               MAPA_BAIRROS=MAPA_BAIRROS,
               )

"""

"result = await limpando_dados(name_arquivo_zap = f'{cidade}_zap_*.json', \n               name_arquivo_vivareal = f'{cidade}_vivareal_*.json', \n               name_arquivo_chave_mao = f'{cidade}_chave_mao_*.json',\n               name_arquivo_olx = f'{cidade}_olx_*.json',\n               name_arquivo_saida = f'{cidade}_imoveis_limpo', \n               pasta_dados = PASTA_DADOS, \n               tipo_async = True,\n               batch = BATCH, \n               cidade_limpeza=cidade_limpeza,\n               cidade_localizacao=cidade_localizacao,\n               estado_limpeza=estado_limpeza,\n               estado_localizacao=estado_localizacao, \n               MAPA_BAIRROS=MAPA_BAIRROS,\n               )\n\n"

In [11]:
def criar_area_ranges(inicio_total: int, fim_total: int, regras_intervalo: list):
    """
    Cria dicionário de ranges seguindo a lógica: inicio = fim_anterior + 1.
    
    regras_intervalo: Lista de tuplas (ate_qual_area, tamanho_do_passo)
    Ex: [(100, 5), (500, 50)] -> Ate 100m² pula de 5 em 5. Ate 500m² pula de 50 em 50.
    """
    ranges = {}
    atual = inicio_total
    
    # Ordena as regras pelo limite de área para garantir a lógica
    regras_intervalo.sort(key=lambda x: x[0])
    
    for limite, passo in regras_intervalo:
        while atual <= limite and atual < fim_total:
            inicio = atual
            fim = atual + passo
            
            # Garante que não ultrapasse o limite atual da regra nem o fim total
            if fim > limite:
                fim = limite
            if fim > fim_total:
                fim = fim_total
                
            ranges[str(inicio)] = str(fim)
            
            # Regra: Próximo inicio = fim anterior + 1
            atual = fim + 1
            
    # Caso o fim_total seja muito grande (o "infinito" da busca)
    # Adicionamos o último range manualmente se ainda não chegamos lá
    if atual <= fim_total:
        ranges[str(atual)] = str(fim_total)
        
    return ranges

# --- EXEMPLO DE USO ---

# Definimos que:
# 1. De 0 a 100: intervalos de 10 em 10
# 2. De 101 a 500: intervalos de 50 em 50
# 3. De 501 a 2000: intervalos de 250 em 250
# 4. Acima disso: até o "infinito"
config_intervalos = [
    (60, 5),
    (150,2),
    (300, 10),
    (500, 20),
    (1000, 50),
    (2000, 1000),
   
]

area_ranges = criar_area_ranges(
    inicio_total=0, 
    fim_total=30000000, 
    regras_intervalo=config_intervalos
)

area_ranges


{'0': '5',
 '6': '11',
 '12': '17',
 '18': '23',
 '24': '29',
 '30': '35',
 '36': '41',
 '42': '47',
 '48': '53',
 '54': '59',
 '60': '60',
 '61': '63',
 '64': '66',
 '67': '69',
 '70': '72',
 '73': '75',
 '76': '78',
 '79': '81',
 '82': '84',
 '85': '87',
 '88': '90',
 '91': '93',
 '94': '96',
 '97': '99',
 '100': '102',
 '103': '105',
 '106': '108',
 '109': '111',
 '112': '114',
 '115': '117',
 '118': '120',
 '121': '123',
 '124': '126',
 '127': '129',
 '130': '132',
 '133': '135',
 '136': '138',
 '139': '141',
 '142': '144',
 '145': '147',
 '148': '150',
 '151': '161',
 '162': '172',
 '173': '183',
 '184': '194',
 '195': '205',
 '206': '216',
 '217': '227',
 '228': '238',
 '239': '249',
 '250': '260',
 '261': '271',
 '272': '282',
 '283': '293',
 '294': '300',
 '301': '321',
 '322': '342',
 '343': '363',
 '364': '384',
 '385': '405',
 '406': '426',
 '427': '447',
 '448': '468',
 '469': '489',
 '490': '500',
 '501': '551',
 '552': '602',
 '603': '653',
 '654': '704',
 '705': '755',
 

In [12]:

area_ranges = criar_area_ranges(
    inicio_total=701,
    fim_total=30000000,
    regras_intervalo=[
        (1500, 150),
        (10000, 1000),
    ]
)
area_ranges

{'701': '851',
 '852': '1002',
 '1003': '1153',
 '1154': '1304',
 '1305': '1455',
 '1456': '1500',
 '1501': '2501',
 '2502': '3502',
 '3503': '4503',
 '4504': '5504',
 '5505': '6505',
 '6506': '7506',
 '7507': '8507',
 '8508': '9508',
 '9509': '10000',
 '10001': '30000000'}

In [51]:
import pandas as pd
import streamlit as st
from pathlib import Path

cidade_nome = 'Florianopolis'

prefixo_arquivo = 'florianopolis'

cidade_pth = 'florianopolis'

BASE_DIR = Path.cwd().parent
pasta = BASE_DIR / 'dados' / cidade_pth

arquivos = "florianopolis_imoveis_limpo_chave_mao_2026-05.parquet"

df = pd.read_parquet(pasta / arquivos)


In [52]:
df['bairro'].unique()

array(['pantanal', 'jardim atlantico', 'cacupe', 'praia brava',
       'trindade', 'estreito', 'agronomica', 'cachoeira do bom jesus',
       'centro', 'coqueiros', 'sao joao do rio vermelho', 'saco grande',
       'itacorubi', 'jurere', 'jurere internacional', 'campeche',
       'joao paulo', 'abraao', 'corrego grande', 'ingleses',
       'ingleses do rio vermelho', 'rio tavares', 'capoeiras',
       'canasvieiras', 'lagoa da conceicao', 'carvoeira',
       'saco dos limoes', 'monte verde', 'morro das pedras', 'balneario',
       'santo antonio de lisboa', 'ribeirao da ilha', 'pantano do sul',
       'santa monica', 'vargem do bom jesus', 'coloninha',
       'vargem grande', 'carianos', 'ponta das canas', 'barra da lagoa',
       'santinho', 'bom abrigo', 'tapera', 'sambaqui', 'novo campeche',
       'itaguacu', 'alto ribeirao', 'alto ribeirao leste', 'jose mendes',
       'ratones', 'vargem pequena', 'moenda', 'morro da cruz', 'daniela',
       'costeira do pirajubae', 'praia da lago

In [53]:
MAPA_BAIRROS = {
        'jurere internacional': ['jurere internacional', 'forte'],
        'jurere': ['jurere tradicional', 'jurere'],
        'ingleses': ['praia dos ingleses', 'ingleses norte', 'sc 403 km 1', 'ingleses'],
        'pantano do sul': ['acores', 'pantano do sul', 'praia da solidao'],
        'estreito': ['canto', 'balneario do estreito'],
        'centro': ['centro', 'beira mar','monte cristo', 'campinas'],
        'lagoa da conceicao': ['porto da lagoa', 'canto da lagoa','costa da lagoa', 'lagoa'],
        'ribeirao da ilha' : ['portal do ribeirao', 'ribeirao da ilha', 'alto ribeirao'],
        'itacorubi' : ['parque sao jorge', 'itacorubi'],
        'canasvieiras' : ['canajure'],
        'barra da lagoa' : ['praia mole'],
        'vargem grande': ['vargem pequena', 'vargem do bom jesus', 'real parque'],
        'rio vermelho': ['sao joao do rio vermelho', 'moenda', 'muquem', 'rio vermelho', 'praia mocambique'],
        'corrego grande' : ['jardim anchieta'],
        'jose mendes' : ['prainha'],
        'balneario' : ['ponta do leal'],
        'ponta das canas' : ['praia da lagoinha', 'lagoinha do norte'],
        'coqueiros' : ['coqueiros'],
        'joao paulo' : ['joao paulo'],
        'agronomica' :['morro da cruz'],
        'tapera da base' : ['tapera']
    }


if MAPA_BAIRROS:
    df['bairro'] = df['bairro'].apply(normalizar_bairros, args=(MAPA_BAIRROS,))

In [54]:
df['bairro'].value_counts()

bairro
ingleses                   6347
centro                     5777
campeche                   3632
estreito                   2372
itacorubi                  1912
trindade                   1672
jurere                     1665
jurere internacional       1639
cachoeira do bom jesus     1607
canasvieiras               1556
agronomica                 1546
joao paulo                 1452
rio vermelho               1426
lagoa da conceicao         1376
coqueiros                  1255
cacupe                     1166
corrego grande             1016
capoeiras                   938
rio tavares                 927
jardim atlantico            887
ribeirao da ilha            744
saco grande                 721
vargem grande               711
carvoeira                   574
pantano do sul              560
balneario                   553
abraao                      527
santo antonio de lisboa     467
morro das pedras            459
saco dos limoes             346
praia brava                 332
p

In [ ]:
df.to_parquet(pasta / arquivos)

: 